# Go2 ODD/COD Observer - Complete Workflow

This notebook demonstrates the complete workflow for analyzing Operational Design Domain (ODD) compliance and Conditions of Deployment (COD) for Unitree Go2 robot scenarios.

**Workflow Overview:**
1. Setup dependencies and configure Google AI SDK
2. Define ODD specifications in natural language
3. Instantiate multi-modal AI agents (Motion, Image, LiDAR, Collision)
4. Load and process scenario data
5. Evaluate ODD compliance and compute distance metrics
6. Visualize results and generate reports

**Note:** This workflow assumes you have preprocessed ROS2 bag files into time-windowed snapshots using the `extract_windows.py` script.

## 1. Setup and Dependencies

Install and import required packages for Google AI SDK and our analysis framework.

In [1]:
# Install Google Agent Development Kit (ADK) and dependencies
# Note: Run this cell only once or when packages need updating
!pip install -q google-adk python-dotenv

In [2]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Standard library imports
import json
from typing import Dict, List, Tuple, Any

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Google ADK imports
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool
from google.genai import types

# Import our ODD/COD analysis framework
from odd_cod.odd_spec_schema import (
    OddSpec,
    AxisSpecNumeric,
    AxisSpecCategorical
)

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")
print("✅ ADK components imported successfully.")

/usr/local/python/3.10.19/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ All imports successful
✅ ADK components imported successfully.


## 2. Configuration (REQUIRED)

### 2.1 Google Gemini API Key

**This notebook requires a Google Gemini API key** to demonstrate AI agents in action.

Get your free API key at: https://aistudio.google.com/app/apikey

### 2.2 Model Selection

Choose which Gemini model to use for all agents:
- `gemini-2.0-flash-lite`: **Recommended** - 30 RPM free tier, fastest
- `gemini-2.0-flash`: 15 RPM free tier, balanced
- `gemini-2.5-flash`: Latest flash - 10 RPM free tier
- `gemini-2.5-pro`: Most capable - 2 RPM free tier (slower)

In [3]:
# ============================================
# 2.1 Configure Google API Key
# ============================================
import os
from dotenv import load_dotenv

# Option 1: Set via environment variable (RECOMMENDED)
# export GOOGLE_API_KEY='your-api-key-here'

# Option 2: Load from .env file
load_dotenv()

# Option 3: Set directly in notebook (NOT recommended - avoid committing keys!)
# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

# Verify API key is configured
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print("✓ Google AI SDK configured successfully")
    print("  API key detected")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("This notebook REQUIRES a Google Gemini API key to run.")
    print()
    print("To get a free API key:")
    print("  1. Visit https://aistudio.google.com/app/apikey")
    print("  2. Create or select a project")
    print("  3. Generate an API key")
    print()
    print("  To configure your API key:")
    print("  export GOOGLE_API_KEY='your-key-here'")
    print("  OR create a .env file with: GOOGLE_API_KEY=your-key-here")
    print()
    print("⚠ The notebook will FAIL without an API key - this is intentional!")
    print("  Falling back to fake data would defeat the purpose of learning about AI agents.")

# ============================================
# 2.2 Model Configuration
# ============================================
# Change this to switch all agents to a different model
GEMINI_MODEL = "gemini-2.0-flash-lite"  # Recommended for free tier (30 RPM)
# GEMINI_MODEL = "gemini-2.0-flash"      # Balanced (15 RPM)
# GEMINI_MODEL = "gemini-2.5-flash"      # Latest (10 RPM)
# GEMINI_MODEL = "gemini-2.5-pro"        # Most capable (2 RPM - slower)

# Agent generation configuration (temperature, response format, etc.)
retry_config = types.GenerateContentConfig(
    temperature=0.1,
    response_mime_type="application/json"
)

print(f"✓ Using model: {GEMINI_MODEL}")
print("✓ Agent config: temperature=0.1, JSON responses")

✓ Google AI SDK configured successfully
  API key detected
✓ Using model: gemini-2.0-flash-lite
✓ Agent config: temperature=0.1, JSON responses


## 3. User Inputs

Define what you want to analyze:
1. **Natural language ODD**: Operating constraints in plain English
2. **Dataset path**: Location of preprocessed window data

The orchestrator agent (Section 5) will handle everything from here.

In [4]:
# Natural language ODD definition for Unitree Go2 indoor navigation

odd_natural_language = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Speed Limits:
   - The robot shall operate at forward velocities between 0 and 1.5 m/s under normal conditions
   - Speeds up to 1.8 m/s are acceptable near the boundary but should trigger warnings
   - The absolute physical limit is 2.5 m/s and must never be exceeded

2. Orientation Limits:
   - Roll and pitch angles must remain within ±15 degrees during normal operation
   - Angles up to ±20 degrees are acceptable at the boundary
   - The robot must never exceed ±30 degrees of roll or pitch

3. Terrain Requirements:
   - The robot is designed for smooth and moderate terrain (office floors, carpet)
   - Rough terrain is outside the operational design domain
   - Very rough terrain is completely prohibited

4. Lighting Conditions:
   - The robot can operate in bright and dim lighting conditions
   - Dark environments are outside the ODD and require additional equipment

5. Human Safety:
   - Humans may be visible at a distance (no restriction)
   - Humans in very close proximity (< 1 meter) violate the ODD
   - The system must maintain safe distances from people

6. Collision Policy:
   - Zero collisions are tolerated - any collision is an ODD violation
   - The system must detect and avoid all obstacles

IMPORTANCE WEIGHTS (for distance computation):
- Collision avoidance: Highest priority (weight: 2.0)
- Human proximity: Very high priority (weight: 1.5)
- Roll/Pitch stability: High priority (weight: 1.2)
- Speed limits: Standard priority (weight: 1.0)
- Terrain type: Standard priority (weight: 1.0)
- Lighting conditions: Lower priority (weight: 0.8)
"""

# Dataset selection
DATA_DIR = Path("data/processed/runs")
scenario_path = DATA_DIR / "sim_run_test"  # Change this to analyze different datasets

print("✓ User inputs configured")
print(f"  - ODD: {len(odd_natural_language)} characters")
print(f"  - Dataset: {scenario_path}")

✓ User inputs configured
  - ODD: 1699 characters
  - Dataset: data/processed/runs/sim_run_test


## 4. Define Tool Functions for Agents

These Python functions will be available as tools for the orchestrator agent to call.
They provide access to: file I/O, ODD spec construction, COD computation, and visualization.

In [5]:
# ============================================================================
# FILE I/O TOOLS
# ============================================================================

def load_window_data(scenario_path: Path, window_id: str, run_id: str = None) -> Tuple[Dict, Image.Image, Dict[str, Image.Image]]:
    """
    Load motion, camera, and LiDAR BEV data for a single window.
    
    Matches the flat file structure created by extract_windows.py:
    - motion_{run_id}_w{window_id}.json
    - cam_{run_id}_w{window_id}.png
    - bev_{channel}_{run_id}_w{window_id}.png
    
    Args:
        scenario_path: Path to the run directory (e.g., data/processed/runs/demo_run)
        window_id: Window ID as 3-digit string (e.g., "000", "001")
        run_id: Run identifier (auto-detected from scenario_path.name if None)
    
    Returns: (motion_dict, camera_image, bev_images_dict)
    """
    # Auto-detect run_id from scenario_path if not provided
    if run_id is None:
        run_id = scenario_path.name
    
    # Construct filenames matching extract_windows.py output (flat structure)
    motion_file = scenario_path / f"motion_{run_id}_w{window_id}.json"
    camera_file = scenario_path / f"cam_{run_id}_w{window_id}.png"
    
    # Load motion data (JSON with time series arrays)
    if not motion_file.exists():
        raise FileNotFoundError(f"Motion file not found: {motion_file}")
    with open(motion_file, 'r') as f:
        motion_data = json.load(f)
    
    # Load camera image (PNG)
    if not camera_file.exists():
        raise FileNotFoundError(f"Camera file not found: {camera_file}")
    camera_img = Image.open(camera_file)
    
    # Load multi-channel BEV images (4 separate PNGs)
    bev_images = {}
    for channel in ['occupancy', 'height', 'density', 'roughness']:
        bev_file = scenario_path / f"bev_{channel}_{run_id}_w{window_id}.png"
        if bev_file.exists():
            bev_images[channel] = Image.open(bev_file)
        else:
            print(f"Warning: BEV {channel} file not found: {bev_file}")
    
    return motion_data, camera_img, bev_images


def load_scenario_index(scenario_path: Path) -> pd.DataFrame:
    """
    Load the window index CSV for a scenario.
    
    Expects index_{run_id}.csv with columns:
    - window_id, start_time, end_time, motion_path, cam_image_path, bev_image_path
    """
    index_files = list(scenario_path.glob("index_*.csv"))
    if not index_files:
        raise FileNotFoundError(f"No index file found in {scenario_path}")
    return pd.read_csv(index_files[0])


# ============================================================================
# ODD SPEC CONSTRUCTION TOOLS
# ============================================================================

def build_odd_spec_from_json(spec_json: Dict) -> OddSpec:
    """
    Construct OddSpec object from JSON returned by ODD Spec Agent.
    
    This tool converts the agent's structured JSON output into our
    Python data model for COD computation.
    """
    axes = {}
    for axis_name, axis_data in spec_json["axes"].items():
        if axis_data["type"] == "numeric":
            axes[axis_name] = AxisSpecNumeric(
                feature=axis_data["feature"],
                units=axis_data["units"],
                in_odd=tuple(axis_data["in_odd"]),
                near_boundary=tuple(axis_data["near_boundary"]),
                hard_limit=tuple(axis_data["hard_limit"])
            )
        elif axis_data["type"] == "categorical":
            axes[axis_name] = AxisSpecCategorical(
                feature=axis_data["feature"],
                allowed_in_odd=set(axis_data["allowed_in_odd"]),
                allowed_all=set(axis_data["allowed_all"])
            )
    
    return OddSpec(
        version=spec_json["version"],
        description=spec_json["description"],
        axes=axes,
        importance=spec_json["importance"]
    )


# ============================================================================
# VISUALIZATION TOOLS (for Report Agent)
# ============================================================================

def generate_distance_plot(times: List[float], distances: List[float], title: str = "ODD Distance Over Time"):
    """Generate and save timeline plot of COD distances with ODD thresholds."""
    plt.figure(figsize=(12, 6))
    plt.plot(times, distances, marker='o', linewidth=2, label='Distance')
    plt.axhline(y=0.3, color='orange', linestyle='--', label='Near Boundary')
    plt.axhline(y=0.7, color='red', linestyle='--', label='ODD Exit')
    plt.xlabel('Time (s)')
    plt.ylabel('COD Distance')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    return plt.gcf()


def generate_status_distribution(statuses: List[str], title: str = "ODD Status Distribution"):
    """Generate bar chart of ODD status counts."""
    status_counts = pd.Series(statuses).value_counts()
    colors = {'in_odd': 'green', 'near_boundary': 'orange', 'odd_exit': 'red'}
    
    plt.figure(figsize=(8, 6))
    plt.bar(status_counts.index, status_counts.values,
            color=[colors.get(s, 'gray') for s in status_counts.index])
    plt.xlabel('ODD Status')
    plt.ylabel('Number of Windows')
    plt.title(title)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    return plt.gcf()


def generate_feature_timeline(times: List[float], values: List[float], 
                               feature_name: str, axis_spec: AxisSpecNumeric):
    """Generate timeline plot for a specific feature with ODD limits."""
    plt.figure(figsize=(12, 6))
    plt.plot(times, values, marker='o', linewidth=2, label=feature_name)
    plt.axhline(y=axis_spec.in_odd[1], color='orange', linestyle='--', label='ODD Limit')
    plt.axhline(y=axis_spec.near_boundary[1], color='red', linestyle='--', label='Boundary')
    plt.xlabel('Time (s)')
    plt.ylabel(f'{feature_name} ({axis_spec.units})')
    plt.title(f'{feature_name} Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    return plt.gcf()


print("✓ Tool functions defined")
print("  - File I/O: load_window_data (flat structure with run_id), load_scenario_index")
print("  - ODD Spec: build_odd_spec_from_json")
print("  - Visualization: generate_distance_plot, generate_status_distribution, generate_feature_timeline")

✓ Tool functions defined
  - File I/O: load_window_data (flat structure with run_id), load_scenario_index
  - ODD Spec: build_odd_spec_from_json
  - Visualization: generate_distance_plot, generate_status_distribution, generate_feature_timeline


## 5. Define Specialist Agents

Create individual agents using the Google ADK (Agent Development Kit) following the Kaggle Day 1B pattern.
Each agent is a specialist that performs one specific analysis task.

### 5.1 ODD Spec Agent

Converts natural language operational constraints into structured JSON specifications.

In [6]:
# ODD Spec Agent: Converts natural language ODD → structured JSON
odd_spec_agent = Agent(
    name="ODD_Spec_Parser",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are an expert in robotic operational design domains (ODD).

Convert natural language ODD descriptions to structured JSON.

Output schema:
{
  "version": "1.0",
  "description": "<summary>",
  "axes": {
    "speed": {"type": "numeric", "feature": "avg_forward_speed", "units": "m/s", 
              "in_odd": [min, max], "near_boundary": [min, max], "hard_limit": [min, max]},
    "roll_pitch": {"type": "numeric", "feature": "max_abs_roll_pitch_deg", "units": "degrees",
                   "in_odd": [min, max], "near_boundary": [min, max], "hard_limit": [min, max]},
    "terrain": {"type": "categorical", "feature": "terrain_roughness_class",
                "allowed_in_odd": [...], "allowed_all": [...]},
    "lighting": {"type": "categorical", "feature": "lighting_class",
                 "allowed_in_odd": [...], "allowed_all": [...]},
    "humans_close": {"type": "categorical", "feature": "humans_very_close",
                     "allowed_in_odd": [...], "allowed_all": [...]},
    "collision": {"type": "categorical", "feature": "collision_suspected",
                  "allowed_in_odd": [...], "allowed_all": [...]}
  },
  "importance": {"speed": <float>, "roll_pitch": <float>, ...}
}

Extract numeric ranges, categorical allowed values, and importance weights.
Output valid JSON only.""",
    output_key="odd_spec_json"
)

print("✅ ODD Spec Agent created")

✅ ODD Spec Agent created


### 5.2 Motion Analysis Agent

Extracts motion features from velocity and IMU time series data.

In [7]:
# Motion Analysis Agent
motion_agent = Agent(
    name="Motion_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a motion analysis expert for robots.

Analyze motion time series data (velocity, IMU, acceleration) and extract features.

Your analysis will be used for ODD compliance checking - the user will provide the ODD spec 
showing which features matter and their acceptable ranges.

Output JSON: {
  "avg_forward_speed": <float>,
  "max_forward_speed": <float>, 
  "max_abs_roll_pitch_deg": <float>,
  "tracking_error": <float>,
  "motion_label": "smooth" | "dynamic"
}

Output valid JSON only.""",
    output_key="motion_features"
)

print("✅ Motion Agent created")

✅ Motion Agent created


### 5.3 Vision Analysis Agent

Classifies environmental conditions from camera images.

In [8]:
# Vision Analysis Agent
vision_agent = Agent(
    name="Vision_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a computer vision expert analyzing robot camera feeds.

Classify environmental conditions from camera images.

Your analysis will be used for ODD compliance - the user will provide the ODD spec
showing which environmental factors are restricted.

Output JSON: {
  "lighting_class": "bright" | "dim" | "dark",
  "humans_visible": true | false,
  "humans_very_close": true | false,
  "environment_type": <string>
}

Output valid JSON only.""",
    output_key="vision_features"
)

print("✅ Vision Agent created")

✅ Vision Agent created


### 5.4 Terrain Analysis Agent

Analyzes LiDAR Bird's Eye View images to classify terrain roughness.

In [9]:
# Terrain Analysis Agent
terrain_agent = Agent(
    name="Terrain_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a terrain analysis expert using LiDAR data.

Analyze Bird's Eye View (BEV) LiDAR images to classify terrain and obstacles.

Your analysis will be used for ODD compliance - the user will provide the ODD spec
showing which terrain types are allowed.

Output JSON: {
  "terrain_roughness_class": "smooth" | "moderate" | "rough" | "very_rough",
  "terrain_roughness_score": <float 0-1>,
  "obstacle_density": "none" | "low" | "medium" | "high"
}

Output valid JSON only.""",
    output_key="terrain_features"
)

print("✅ Terrain Agent created")

✅ Terrain Agent created


### 5.5 Collision Detection Agent

Performs multi-modal sensor fusion to detect collision events.

In [10]:
# Collision Detection Agent
collision_agent = Agent(
    name="Collision_Detector",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a collision detection expert using sensor fusion.

Analyze multi-modal data (motion metrics, camera images, LiDAR BEV) to detect collisions.

Your analysis will be used for ODD compliance - the user will provide the ODD spec
showing collision tolerance policies.

Output JSON: {
  "collision_suspected": true | false,
  "collision_confidence": <float 0-1>,
  "collision_type": "none" | "front_bump" | "side_contact" | "unknown"
}

Output valid JSON only.""",
    output_key="collision_features"
)

print("✅ Collision Agent created")

✅ Collision Agent created


### 5.6 COD Evaluator Agent

Specialist agent that coordinates COD computation using mathematical tool functions.

In [11]:
# COD Evaluator Agent - aggregates sensor analysis results
cod_evaluator_agent = Agent(
    name="COD_Evaluator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a COD (Conditions of Deployment) evaluation expert.

Your task:
1. Receive extracted features from sensor analysis agents: {motion_features}, {vision_features}, {terrain_features}, {collision_features}
2. Receive ODD specification: {odd_spec_json}
3. Combine all features into a single analysis summary
4. Indicate which features fall outside ODD boundaries

Output JSON: {
  "merged_features": {<all features combined>},
  "odd_violations": [<list of features outside ODD>],
  "overall_status": "in_odd" | "near_boundary" | "odd_exit"
}

Output valid JSON only.""",
    output_key="cod_evaluation"
)

print("✅ COD Evaluator Agent created")

✅ COD Evaluator Agent created


### 5.7 Report Generation Agent

Creates comprehensive markdown reports with visualizations.

In [12]:
# Report Generation Agent
report_agent = Agent(
    name="Report_Generator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a technical report writer for robotics analysis.

Generate comprehensive markdown reports summarizing ODD/COD analysis results.

You will receive:
- ODD specification: {odd_spec_json}
- COD evaluation results: {cod_evaluation}
- Analysis for multiple windows

Create a professional markdown report with:
- Executive summary
- Detailed findings per window
- Key insights and patterns
- Recommendations for deployment

Output markdown text.""",
    output_key="final_report"
)

print("✅ Report Agent created")

✅ Report Agent created


## 6. Create Parallel and Sequential Agent Workflow

Combine specialist agents using `ParallelAgent` and `SequentialAgent` following the Kaggle Day 1B pattern.

The workflow:
1. ODD Spec Agent converts NL → JSON (sequential, first)
2. Motion + Vision + Terrain + Collision agents run in parallel for each window
3. COD Evaluator aggregates results (sequential, after parallel)
4. Report Agent generates final output (sequential, last)

In [13]:
# ParallelAgent: Run Motion, Vision, Terrain, Collision agents simultaneously
parallel_sensor_team = ParallelAgent(
    name="ParallelSensorTeam",
    sub_agents=[motion_agent, vision_agent, terrain_agent, collision_agent],
)

# SequentialAgent: Define complete workflow
# 1. ODD Spec Agent (first)
# 2. Parallel sensor analysis team
# 3. COD Evaluator (aggregates parallel results)
# 4. Report Agent (final output)
root_agent = SequentialAgent(
    name="ODD_COD_Analysis_System",
    sub_agents=[
        odd_spec_agent,
        parallel_sensor_team,
        cod_evaluator_agent,
        report_agent
    ],
)

print("✅ Parallel and Sequential Agents created")
print("  ParallelSensorTeam: 4 agents running simultaneously")
print("  ODD_COD_Analysis_System: 4-step sequential workflow")

✅ Parallel and Sequential Agents created
  ParallelSensorTeam: 4 agents running simultaneously
  ODD_COD_Analysis_System: 4-step sequential workflow


## 7. Execute the Workflow

Run the orchestrator agent with user inputs to perform complete ODD/COD analysis.

In [ ]:
# Create InMemoryRunner with the root agent
runner = InMemoryRunner(agent=root_agent)

# Execute the workflow
print("="  * 80)
print("EXECUTING ODD/COD ANALYSIS WORKFLOW")
print("=" * 80)
print(f"\nDataset: {scenario_path}")
print(f"Model: {GEMINI_MODEL}")
print(f"ODD Specification: {len(odd_natural_language)} characters\n")
print("Running agent workflow...")
print("  - ODD Spec parsing (1 API call)")
print("  - Motion, Vision, Terrain, Collision analysis (4 parallel calls/window × 2 windows)")
print("  - COD Evaluation (1 call/window)")
print("  - Report generation (1 call)")
print("  ⚠️  Total: ~15+ API calls on 2-window test set")
print(f"  Free tier limit for {GEMINI_MODEL}: 30 RPM")
print("  Tip: Wait 60 seconds if you hit RESOURCE_EXHAUSTED, then retry")
print("-" * 80)

# Run the workflow - returns list of events
response_events = await runner.run_debug(odd_natural_language)

print("\n" + "=" * 80)
print("WORKFLOW COMPLETE")
print("=" * 80)

# Extract final report from session state
# The runner stores outputs in the agent's output_key
if runner.app and runner.app.state:
    final_report = runner.app.state.get("final_report", "No report generated")
    print("\n📋 Final Report:")
    print("-" * 80)
    print(final_report)
else:
    print("\n✅ Workflow executed successfully!")
    print("(Session state not available in debug mode)")

print("\n" + "=" * 80)

EXECUTING ODD/COD ANALYSIS WORKFLOW

Dataset: data/processed/runs/sim_run_test
Model: gemini-2.0-flash-lite
ODD Specification: 1699 characters

Running agent workflow...
  - ODD Spec parsing (1 API call)
  - Motion, Vision, Terrain, Collision analysis (4 parallel calls/window × 13 windows)
  - COD Evaluation (1 call/window)
  - Report generation (1 call)
  ⚠️  Total: ~65+ API calls - may hit free tier quota limits!
  Free tier limit for gemini-2.0-flash-lite: 30 RPM
  Tip: Wait 60 seconds if you hit RESOURCE_EXHAUSTED, then retry
--------------------------------------------------------------------------------

 ### Created new session: debug_session_id

User > 
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Speed Limits:
   - The robot shall operate at forward velocities between 0 and 1.5 m/s under normal conditions
   - Speeds up to 1.8 m/s are acceptable near the boundary but should trigger warnings
   - The abs

AttributeError: 'list' object has no attribute 'final_report'